# ImageJ image projection and PNG export

A reusable Fiji/ImageJ workflow for Bio-Formats-compatible microscopy images.

At the start, the notebook imports an image that you choose and confirms that Fiji is available. You then select channels and z-slices, crop the field of view, and adjust display ranges. The workflow produces three useful outputs beside the input image: an uncompressed OME-TIFF copy of the imported volume, a maximum-intensity projection with adjusted channels and a scale bar, and a final RGB PNG for easy sharing or presentation. Values that are likely to change between images are grouped in the tuning cells below.

In [ ]:
# ⚙️ Start the Fiji/ImageJ bridge
import os
from pathlib import Path

os.environ.setdefault("DISPLAY", ":1")

import imagej

FIJI_PATH = Path("/home/jovyan/Fiji.app")
ij = imagej.init(str(FIJI_PATH), mode="interactive")

In [ ]:
# ✅ Confirm that Fiji is available
ij.ui().showUI()
print(f"ImageJ version: {ij.getVersion()}")

## Restart the image workflow

If you are rerunning the notebook with the same kernel, use the next cell to close all images currently open in Fiji. This keeps ImageJ loaded and lets you continue from the workflow steps below.

In [ ]:
# 🧹 Close open Fiji images without restarting ImageJ
ij.py.run_macro('run("Close All");')
print("All open Fiji images closed; ImageJ is still available.")

## Tune the workflow

Edit the following cells for the image you want to analyze. Coordinates and dimensions are in pixels; z and channel indices follow Bio-Formats/ImageJ conventions. The default values are examples and may need to be changed for each image.

In [ ]:
# Choose the input image
# Replace this with the path to any Bio-Formats-compatible image.
file_path = Path("your_image_file.nd2")
input_path = file_path.expanduser().resolve()

if not input_path.is_file():
    raise FileNotFoundError(f"Image not found: {input_path}")

output_dir = input_path.parent
output_tif = output_dir / f"{input_path.stem}.ome.tif"
output_png = output_dir / f"{input_path.stem}.png"
print(f"Input: {input_path}")

### 🧬 Tune and import the acquisition range

Channels and z-slices are adjusted inside the execution cell below.

In [ ]:
# 🧬 Optionally select channels and z-slices
# Leave any of these variables undefined, or set them to None, to omit that option.
range_values = {
    "c_begin": globals().get("channel_start"),
    "c_end": globals().get("channel_end"),
    "c_step": globals().get("channel_step"),
    "z_begin": globals().get("z_begin"),
    "z_end": globals().get("z_end"),
    "z_step": globals().get("z_step"),
}
range_args = " ".join(
    f"{name}={value}"
    for name, value in range_values.items()
    if value is not None
)

import_options = (
    f"open=[{input_path.as_posix()}] autoscale color_mode=Default "
    f"rois_import=[ROI manager] view=Hyperstack stack_order=XYCZT"
)
if range_args:
    import_options = f"{import_options} specify_range {range_args}"

import_macro = f'''run("Bio-Formats Importer", "{import_options}");'''
ij.py.run_macro(import_macro)
print(f"Imported image with options: {range_args or 'full acquisition'}")

### ✂️ Tune and crop the merged image

In [ ]:
# ✂️ Tune and apply the crop rectangle
# Coordinates and dimensions are in pixels.
crop_x = 0
crop_y = 308
crop_width = 2388
crop_height = 1724

crop_macro = f"makeRectangle({crop_x}, {crop_y}, {crop_width}, {crop_height}); run(\"Crop\");"
ij.py.run_macro(crop_macro)
print(f"Crop applied: x={crop_x}, y={crop_y}, width={crop_width}, height={crop_height}")

### 📐 Scale the cropped image

Set the scaling factors for each axis. The target width, height, and depth are calculated automatically from the original image dimensions captured before cropping.

In [ ]:
# 📐 Tune axis scaling factors; target dimensions are derived from the original image
scale_x = 0.2
scale_y = scale_x
scale_z = 1.0

scaled_width = round(original_width * scale_x)
scaled_height = round(original_height * scale_y)
scaled_depth = round(original_depth * scale_z)

scale_macro = (
    f'run("Scale...", "x={scale_x} y={scale_y} z={scale_z} '
    f'width={scaled_width} height={scaled_height} depth={scaled_depth} '
    'interpolation=Bilinear average create");'
)
ij.py.run_macro(scale_macro)
print(
    f"Scaled image from {original_width} × {original_height} × {original_depth} "
    f"to {scaled_width} × {scaled_height} × {scaled_depth}"
)

In [ ]:
# 💾 Optionally save the scaled image as a new OME-TIFF
save_scaled_image = False
scaled_output_tif = output_dir / f"{input_path.stem}_rescaled.ome.tif"

if save_scaled_image:
    scaled_export_macro = (
        f'''run("Bio-Formats Exporter", "save=[{scaled_output_tif.as_posix()}] '''
        '''export compression=Uncompressed");'''
    )
    ij.py.run_macro(scaled_export_macro)
    print(f"Saved scaled image: {scaled_output_tif}")
else:
    print("Scaled image export skipped")

### 💾 Export the imported volume

This preserves the selected Bio-Formats volume as an uncompressed OME-TIFF.

In [ ]:
# Save the imported stack before projection
export_macro = f'''run("Bio-Formats Exporter", "save=[{output_tif.as_posix()}] export compression=Uncompressed");'''
ij.py.run_macro(export_macro)
print(f"Saved: {output_tif}")

### 🌟 Create a maximum-intensity projection

In [ ]:
# Collapse the selected z-range into one maximum-intensity image
ij.py.run_macro('run("Z Project...", "projection=[Max Intensity]");')
print("Maximum-intensity projection created")

### 🧩 Split the projected channels

In [ ]:
# Split the maximum-intensity projection into one window per channel
ij.py.run_macro('run("Split Channels");')
print("Projected channels split")

### 🟢 Tune channel 1 display range

Set the lower and upper intensity limits for channel 1 independently.

In [ ]:
channel_1_min = 100
channel_1_max = 2000
channel_1_window = f"C1-MAX_{input_path.name}"

channel_1_macro = f'''selectWindow("{channel_1_window}");
setMinAndMax({channel_1_min}, {channel_1_max});'''
ij.py.run_macro(channel_1_macro)
print(f"Channel 1 display range: {channel_1_min}-{channel_1_max}")

### 🔴 Tune channel 2 display range

Set the lower and upper intensity limits for channel 2 independently.

In [ ]:
channel_2_min = 100
channel_2_max = 1000
channel_2_window = f"C2-MAX_{input_path.name}"

channel_2_macro = f'''selectWindow("{channel_2_window}");
setMinAndMax({channel_2_min}, {channel_2_max});'''
ij.py.run_macro(channel_2_macro)
print(f"Channel 2 display range: {channel_2_min}-{channel_2_max}")

### 🧩 Merge the adjusted channels

In [ ]:
# Merge the channels after their display ranges have been adjusted
merge_macro = f'''run("Merge Channels...", "c1={channel_1_window} c2={channel_2_window} create keep");'''
ij.py.run_macro(merge_macro)
print("Adjusted channels merged")

### 📏 Tune and add the scale bar

In [ ]:
# 📏 Tune and add the scale bar
scale_bar_width = 200
scale_bar_height = 100
scale_bar_thickness = 30
scale_bar_font_size = 0

scale_macro = f'''run("Scale Bar...", "width={scale_bar_width} height={scale_bar_height} thickness={scale_bar_thickness} font={scale_bar_font_size} bold overlay");'''
ij.py.run_macro(scale_macro)
print(f"Scale bar added: {scale_bar_width} × {scale_bar_height} px")

### 🖼️ Convert and save the final PNG

In [ ]:
save_macro = f'''run("RGB Color");
saveAs("PNG", "{output_png.as_posix()}");'''
ij.py.run_macro(save_macro)
print(f"Saved final PNG: {output_png}")